In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import f1_score
import copy
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import torch.nn as nn
from sklearn.metrics import precision_score, recall_score
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/OurBalancedDataset.csv")
print(data.shape)
data.head(5)

(26671, 7)


,text,label,multi_label,language,dataset_source,binary_label,char_length
0,"El jueves pasado, 28 de abril, el Tribunal Pen...",1,Mistral-7B-Instruct-v0.2,es,MULTITuDE,machine,1520
1,Für McLaren-Mercedes lief es diese Saison bisl...,0,human,de,MULTITuDE,human,1195
2,Las Vegas: Die gesuchte bzw. evtl bereits gefu...,0,human,de,MultiSocial,human,139
3,"El Ministerio de Industria, Energía y Turismo ...",0,human,es,MULTITuDE,human,1058
4,"Na gut, vielleicht läuft er dann mal im Vollra...",0,human,de,MultiSocial,human,174


In [ ]:
df = data[["text", "label","language"]].copy()
df["label"] = df["label"].astype(int)

df_trval, df_test = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=42 )
df_train,df_val = train_test_split(df_trval, test_size=0.1,stratify=df_trval["label"],random_state=42)
df_trval = df_trval.reset_index(drop=True)
df_test= df_test.reset_index(drop=True)
df_train= df_train.reset_index(drop=True)
df_val= df_val.reset_index(drop=True)
print("Train size:", len(df_train))
print("Val size  :", len(df_val))
print("Test size :", len(df_test))

Train size: 21602
Val size  : 2401
Test size : 2668


In [ ]:
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
num_labels = 2

# MAX_LEN = 512

class HumanvsAI(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "labels": self.labels[idx]
        }

def make_collate_fn(max_len):
    def collate_fn(batch):
        texts = [item["text"] for item in batch]
        labels = [item["labels"] for item in batch]

        enc = tokenizer( texts, padding="longest", truncation=True, max_length=max_len, return_tensors="pt")
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc
    return collate_fn

train_dataset = HumanvsAI(df_train)
val_dataset   = HumanvsAI(df_val)
test_dataset  = HumanvsAI(df_test)

# BATCH_SIZE = 4

def build_loaders(max_len, batch_size):
    train_dataset = HumanvsAI(df_train)
    val_dataset   = HumanvsAI(df_val)

    collate_fn = make_collate_fn(max_len)
    pin = torch.cuda.is_available()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,collate_fn=collate_fn, pin_memory=pin)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=True,collate_fn=collate_fn, pin_memory=pin)
    return train_loader, val_loader

In [ ]:
def train_model(model, train_loader, optimizer, loss_fn, device, scheduler=None):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)
    for batch in train_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")

    return total_loss / len(train_loader), correct / total

In [ ]:
def validate_model(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", leave=False)
        for batch in val_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=-1)

            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

    avg_loss = total_loss / len(val_loader)
    val_acc = (np.array(y_true) == np.array(y_pred)).mean()
    val_f1  = f1_score(y_true, y_pred, average="macro")
    return avg_loss, val_acc, val_f1

In [ ]:
def run_trial(lr, max_len, batch_size, weight_decay=0.01, epochs=3, save_dir="/content/drive/MyDrive/mBERT_best_trial"):
    torch.cuda.empty_cache()

    train_loader, val_loader = build_loaders(max_len, batch_size)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, eps=1e-8)
    num_training_steps = len(train_loader) * epochs
    scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=max(1, num_training_steps)) # To avoid gradient explosion
    loss_fn = torch.nn.CrossEntropyLoss()

    best_val_f1 = -1.0
    best_val_acc = -1.0
    best_epoch = -1

    best_state_dict = None

    for ep in range(epochs):
        print(f"\nTrial(lr={lr}, max_len={max_len}, bs={batch_size}, wd={weight_decay}) Epoch {ep+1}/{epochs}")
        tr_loss, tr_acc = train_model(model, train_loader, optimizer, loss_fn, device, scheduler=scheduler)
        va_loss, va_acc, va_f1 = validate_model(model, val_loader, loss_fn, device)
        print(f"Train: loss={tr_loss:.4f} acc={tr_acc:.4f} | Val: loss={va_loss:.4f} acc={va_acc:.4f} macroF1={va_f1:.4f}")

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_val_acc = va_acc
            best_epoch = ep + 1
            best_state_dict = copy.deepcopy(model.state_dict())

    return best_val_f1, best_val_acc, best_epoch, best_state_dict

In [ ]:
grid = [
    {"lr": 2e-5, "max_len": 256, "batch_size": 4,  "wd": 0.01},
    {"lr": 1e-5, "max_len": 512, "batch_size": 4,  "wd": 0.01},
    {"lr": 3e-5, "max_len": 128, "batch_size": 6, "wd": 0.00},
]

In [ ]:
best_score = -1.0
best_cfg = None
best_state = None
best_epoch = None

for cfg in grid:
    val_f1, val_acc, ep, state = run_trial( cfg["lr"], cfg["max_len"], cfg["batch_size"], cfg["wd"], epochs=3)

    print(f"RESULT cfg={cfg} | best_val_macroF1={val_f1:.4f} | best_val_acc={val_acc:.4f} | best_epoch={ep}\n")

    if val_f1 > best_score:
        best_score = val_f1
        best_cfg = cfg
        best_state = state
        best_epoch = ep

save_path = "/content/drive/MyDrive/mBERT_best_model.pt"
torch.save({
    "model_name": model_name,
    "num_labels": num_labels,
    "best_cfg": best_cfg,
    "best_epoch": best_epoch,
    "state_dict": best_state
}, save_path)

print("BEST CONFIG:", best_cfg)
print(" BEST VAL macro-F1:", best_score)
print(" Saved best model to:", save_path)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Trial(lr=2e-05, max_len=256, bs=4, wd=0.01) Epoch 1/3


Training:   0%|          | 0/5401 [00:00<?, ?it/s]

Validation:   0%|          | 0/601 [00:00<?, ?it/s]

Train: loss=0.2591 acc=0.9337 | Val: loss=0.1326 acc=0.9754 macroF1=0.9754

Trial(lr=2e-05, max_len=256, bs=4, wd=0.01) Epoch 2/3


...

Validation:   0%|          | 0/401 [00:00<?, ?it/s]

Train: loss=0.1474 acc=0.9663 | Val: loss=0.4851 acc=0.9130 macroF1=0.9124

Trial(lr=3e-05, max_len=128, bs=6, wd=0.0) Epoch 3/3


Training:   0%|          | 0/3601 [00:00<?, ?it/s]

Validation:   0%|          | 0/401 [00:00<?, ?it/s]

Train: loss=0.0424 acc=0.9923 | Val: loss=0.5218 acc=0.9304 macroF1=0.9302
RESULT cfg={'lr': 3e-05, 'max_len': 128, 'batch_size': 6, 'wd': 0.0} | best_val_macroF1=0.9374 | best_val_acc=0.9375 | best_epoch=1

BEST CONFIG: {'lr': 1e-05, 'max_len': 512, 'batch_size': 4, 'wd': 0.01}
 BEST VAL macro-F1: 0.9870885696953737
 Saved best model to: /content/drive/MyDrive/mBERT_best_model.pt


In [ ]:
mdl_path = "/content/drive/MyDrive/mBERT_best_model.pt"

mdl_test = torch.load(mdl_path, map_location=device)

model_test = AutoModelForSequenceClassification.from_pretrained(mdl_test["model_name"],num_labels=mdl_test["num_labels"]).to(device)

model_test.load_state_dict(mdl_test["state_dict"])
model_test.eval()

print("Loaded best config:", mdl_test["best_cfg"])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded best config: {'lr': 1e-05, 'max_len': 512, 'batch_size': 4, 'wd': 0.01}


In [ ]:
def build_test_loader(df_test, max_len, batch_size):
    test_dataset = HumanvsAI(df_test.reset_index(drop=True))
    collate = make_collate_fn(max_len)
    pin = torch.cuda.is_available()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate, pin_memory=pin)
    return test_loader

In [ ]:
_max_len = mdl_test["best_cfg"]["max_len"]

test_loader = build_test_loader(df_test, _max_len, 1)
print("Test batches:", len(test_loader))


Test batches: 2668


In [ ]:
loss_fn = nn.CrossEntropyLoss()

def test_eval(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    bar = tqdm(loader, desc="Testing")
    for batch in bar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = loss_fn(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=-1)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return avg_loss, acc, macro_f1, np.array(y_true), np.array(y_pred)

test_loss, test_acc, test_f1, y_true, y_pred = test_eval(model_test, test_loader, loss_fn, device)
print(f"\n Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test Macro-F1: {test_f1:.4f}")

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
print("\nConfusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["human","ai"], digits=4, zero_division=0))

Testing:   0%|          | 0/2668 [00:00<?, ?it/s]


 Test Loss: 0.0901 | Test Acc: 0.9861 | Test Macro-F1: 0.9861

Confusion Matrix:
 [[1308   26]
 [  11 1323]]

Classification Report:
              precision    recall  f1-score   support

       human     0.9917    0.9805    0.9861      1334
          ai     0.9807    0.9918    0.9862      1334

    accuracy                         0.9861      2668
   macro avg     0.9862    0.9861    0.9861      2668
weighted avg     0.9862    0.9861    0.9861      2668



In [ ]:
df_test = df_test.copy()
df_test.columns = df_test.columns.str.strip()
if "language" not in df_test.columns and "lang" in df_test.columns:
    df_test = df_test.rename(columns={"lang": "language"})

df_test["language"] = df_test["language"].astype(str).str.strip()

results = []
langs = sorted(df_test["language"].dropna().unique())

for lang in langs:
    df_lang = df_test[df_test["language"] == lang].copy().reset_index(drop=True)
    if len(df_lang) < 10:
        continue

    test_loader_lang = build_test_loader(df_lang, _max_len, 1)

    loss_l, acc_l,f1_l, yt, yp = test_eval(model_test, test_loader_lang, loss_fn, device)

    results.append({
    "language": lang,
    "n": len(df_lang),
    "loss": loss_l,
    "acc": acc_l,
    "macro_f1": f1_l,
    "precision": precision_score(yt, yp, average="macro", zero_division=0),
    "recall": recall_score(yt, yp, average="macro", zero_division=0),
})

per_lang_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
per_lang_df

Testing:   0%|          | 0/653 [00:00<?, ?it/s]

Testing:   0%|          | 0/633 [00:00<?, ?it/s]

Testing:   0%|          | 0/666 [00:00<?, ?it/s]

Testing:   0%|          | 0/716 [00:00<?, ?it/s]

,language,n,loss,acc,macro_f1,precision,recall
0,de,633,0.020919,0.996840,0.996840,0.996840,0.996840
1,en,666,0.064526,0.989489,0.989475,0.989329,0.989661
2,es,716,0.106884,0.986034,0.985965,0.987113,0.985207
3,cs,653,0.164853,0.972435,0.972430,0.972399,0.972479


In [ ]:
language = ["pt","nl","ru"]
data_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/multitude_v3_clean.csv")
df_testlang = data_test[data_test["language"].isin(language)]
print(df_testlang.shape)
df_testlang.head(5)

(31015, 8)


,Unnamed: 0,text,label,multi_label,split,language,length,source
1,1,В ходе дискуссий стороны договорились расширит...,1,aya-101,test,ru,125,MULTITuDE_MassiveSumm_dw
18,18,MAROKKO: NOG GEEN EINDE AAN DE WODE TEGEN DE S...,1,vicuna-13b,train,nl,239,MULTITuDE_MassiveSumm_globalvoices
19,19,Уже с трудом у меня давали интервью: многие од...,1,opt-iml-max-30b,train,ru,48,MULTITuDE_MassiveSumm_rbc
24,24,Как компании Google предлагают участников встр...,1,opt-iml-max-30b,train,ru,34,MULTITuDE_MassiveSumm_bbc
25,25,.A polícia britânica deteve um suspeito de hom...,1,v5-Eagle-7B-HF,train,pt,250,MULTITuDE_MassiveSumm_rfi


In [ ]:
lang_balance = df_testlang["label"].value_counts()
print("Before:\n", lang_balance)

n = lang_balance.min()

df0 = df_testlang[df_testlang["label"] == 0]
df1 = df_testlang[df_testlang["label"] == 1]

if len(df0) > len(df1):
    df0 = df0.sample(n=n, random_state=42)
else:
    df1 = df1.sample(n=n, random_state=42)

df_unseenlang = ( pd.concat([df0, df1], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True))
print("\nAfter:\n", df_unseenlang["label"].value_counts())

Before:
 label
1    27117
0     3898
Name: count, dtype: int64

After:
 label
0    3898
1    3898
Name: count, dtype: int64


In [ ]:
df_unseenlang = df_unseenlang.copy()
df_unseenlang.columns = df_unseenlang.columns.str.strip()

df_unseenlang["language"] = df_unseenlang["language"].astype(str).str.strip()

langs = sorted(df_unseenlang["language"].dropna().unique())

results = []
for lang in langs:
    df_lang = df_unseenlang[df_unseenlang["language"] == lang].copy().reset_index(drop=True)
    if len(df_lang) < 10:
        continue

    test_loader_lang = build_test_loader(df_lang, _max_len, 1)

    loss_l, acc_l, f1_l, yt, yp = test_eval(model_test, test_loader_lang, loss_fn, device)

    results.append({
        "language": lang,
        "n": len(df_lang),
        "loss": loss_l,
        "acc": acc_l,
        "macro_f1": f1_l,
        "precision": precision_score(yt, yp, average="macro", zero_division=0),
        "recall": recall_score(yt, yp, average="macro", zero_division=0),
    })

per_lang_df = pd.DataFrame(results)
if len(per_lang_df) == 0:
    print("Insufficient data")
else:
    per_lang_df = per_lang_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)

per_lang_df

Testing:   0%|          | 0/2589 [00:00<?, ?it/s]

Testing:   0%|          | 0/2642 [00:00<?, ?it/s]

Testing:   0%|          | 0/2565 [00:00<?, ?it/s]

,language,n,loss,acc,macro_f1,precision,recall
0,nl,2589,0.870279,0.864426,0.864418,0.864458,0.864410
1,pt,2642,1.012871,0.854656,0.854065,0.863370,0.855952
2,ru,2565,1.601041,0.766472,0.754607,0.822040,0.763614
